In [1]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, "/nfs/norasys/notebooks/camaret/ic_segmentation")

from hydra import compose, initialize_config_dir
from src.dataloaders.totalseg2d_shared_dataloader import TotalSeg2DSharedDataset

config_path = "/nfs/norasys/notebooks/camaret/ic_segmentation/configs"
with initialize_config_dir(config_dir=str(config_path), version_base=None):
    cfg = compose(config_name="eval", overrides=["experiment=106_3_lvls"])

# Instantiate dataset with config params (matching scripts/eval.py)
data_dir = Path(cfg.paths.DATA_DIR)
shared_dir = data_dir / f"{cfg.base_dataset}_2d_shared"

# Get image size as tuple
img_size = cfg.preprocessing.image_size
image_size = tuple(img_size[:2]) if isinstance(img_size, (list, tuple)) else (img_size, img_size)

# Handle val_split
val_split_cfg = cfg.get("val_split", ["val", "test"])
val_split = list(val_split_cfg) if hasattr(val_split_cfg, "__iter__") and not isinstance(val_split_cfg, str) else val_split_cfg

# Handle label_ids
val_labels = cfg.val_label_ids if isinstance(cfg.val_label_ids, str) else list(cfg.val_label_ids)

dataset = TotalSeg2DSharedDataset(
    root_dir=str(shared_dir),
    stats_path=str(shared_dir / "stats.pkl"),
    label_id_list=val_labels,
    context_size=cfg.context_size,
    axes=("z", "y", "x"),
    image_size=image_size,
    crop_to_bbox=cfg.preprocessing.crop_to_bbox,
    bbox_padding=cfg.preprocessing.bbox_padding,
    min_coverage=cfg.get("min_coverage", 100),
    min_coverage_ratio=cfg.get("min_coverage_ratio", 0.1),
    split=val_split,
    max_labels=cfg.get("max_labels", None),
    same_case_context=cfg.get("same_case_context", False),
    max_slices_per_group=cfg.get("max_slices_per_group", None),
    slice_selection=cfg.get("slice_selection", "all"),
    random_context=False,
    augment=False,
)

samples = dataset.samples
stats = dataset.stats
print(f"Dataset has {len(samples)} samples")

totalseg_dir: /nfs/data/nii/data1/Analysis/camaret___in_context_segmentation/ANALYSIS_20251122/data/totalseg
Loaded stats for 500 cases
Filtered to 53 cases for split '['val', 'test']'
Building sample index...


Scanning: 100%|██████████| 53/53 [00:00<00:00, 307.73it/s]

Built 28148 samples from 50 labels
Dataset has 28148 samples


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Count samples per (case_id, label_id, axis) from the filtered samples list
# samples structure: [(case_id, label_id, axis, slice_idx), ...]
samples_per_group = Counter((case_id, label_id, axis) for case_id, label_id, axis, _ in samples)

# Example: get slice counts for a specific label/axis
axis = "x"
label = "liver"
covered_slices_per_case = [
    count for (case_id, lbl, ax), count in samples_per_group.items()
    if lbl == label and ax == axis
]
covered_slices_per_case = np.array(covered_slices_per_case)
print(f"{label} on axis {axis}: {len(covered_slices_per_case)} cases, {covered_slices_per_case.sum()} samples")

liver on axis x: 39 cases, 389 samples


In [3]:
# Compute ratio of (case,label,axis) groups with < min_slices_floor samples
from collections import defaultdict

min_slices_floor = 10

for axis in ('z', 'y', 'x'):
    # Count cases per label for this axis using filtered samples
    label_counts = defaultdict(lambda: [0, 0])  # label -> [total_cases, cases_lt_floor]
    
    for (case_id, label_id, ax), num_slices in samples_per_group.items():
        if ax != axis:
            continue
        label_counts[label_id][0] += 1
        if num_slices < min_slices_floor:
            label_counts[label_id][1] += 1

    # Compute ratios and sort by ratio descending
    label_ratios = [
        (label, lt_floor / total, lt_floor, total)
        for label, (total, lt_floor) in label_counts.items()
    ]
    label_ratios.sort(key=lambda x: -x[1])

    print(f"\n=== Axis {axis} ===")
    print(f"{'Label':<35} {'Ratio':>7} {f'<{min_slices_floor}':>6} {'Total':>7}")
    print("-" * 60)
    for label, ratio, lt_floor, total in label_ratios[:20]:
        print(f"{label:<35} {ratio:>7.2%} {lt_floor:>6} {total:>7}")


=== Axis z ===
Label                                 Ratio    <10   Total
------------------------------------------------------------
brachiocephalic_vein_right          100.00%     30      30
superior_vena_cava                   88.24%     30      34
brachiocephalic_trunk                73.33%     22      30
atrial_appendage_left                64.52%     20      31
rib_right_12                         63.16%     12      19
rib_left_12                          54.55%     12      22
esophagus                            31.82%     14      44
iliopsoas_right                      31.25%     10      32
iliopsoas_left                       23.33%      7      30
rib_right_11                         21.21%      7      33
rib_left_11                          18.18%      6      33
rib_left_8                           16.22%      6      37
rib_left_7                           15.79%      6      38
portal_vein_and_splenic_vein         14.71%      5      34
vertebrae_T8                         1

In [ ]:
# Test dataloader WITH augmentation enabled
import yaml
import time

# Load custom augmentation config
aug_config_path = "/nfs/norasys/notebooks/camaret/ic_segmentation/configs/augmentation/custom.yaml"
with open(aug_config_path, 'r') as f:
    aug_config = yaml.safe_load(f)
    
print("Augmentation config:")
print(f"  Type: {aug_config['augmentation'].get('type', 'N/A')}")
print(f"  Medical specialty enabled: {aug_config['augmentation'].get('medical_specialty', {}).get('enabled', False)}")

# Create dataset with augmentation
dataset_aug = TotalSeg2DSharedDataset(
    root_dir=str(shared_dir),
    stats_path=str(shared_dir / "stats.pkl"),
    label_id_list=val_labels,
    context_size=cfg.context_size,
    axes=("z", "y", "x"),
    image_size=image_size,
    crop_to_bbox=cfg.preprocessing.crop_to_bbox,
    bbox_padding=cfg.preprocessing.bbox_padding,
    min_coverage=cfg.get("min_coverage", 100),
    min_coverage_ratio=cfg.get("min_coverage_ratio", 0.1),
    split=val_split,
    max_labels=cfg.get("max_labels", None),
    same_case_context=cfg.get("same_case_context", False),
    max_slices_per_group=cfg.get("max_slices_per_group", None),
    slice_selection=cfg.get("slice_selection", "all"),
    random_context=True,
    augment=True,
    augmentation_config=aug_config['augmentation'],
)

print(f"\nDataset with augmentation: {len(dataset_aug)} samples")
print(f"Augmentation type: {dataset_aug.augmentation_type}")

In [ ]:
# Test fetching samples with augmentation
import random
import time

print("Testing sample fetching with augmentation...")
for i in range(5):
    idx = random.randint(0, len(dataset_aug) - 1)
    t0 = time.time()
    sample = dataset_aug[idx]
    t1 = time.time()
    print(f"Sample {i+1}: idx={idx}, label={sample['label_id']}, time={t1-t0:.2f}s, "
          f"img={sample['image'].shape}, ctx={sample.get('context_in', torch.zeros(0)).shape}")

print("\nTesting DataLoader with workers...")
from torch.utils.data import DataLoader
from src.dataloaders.totalseg2d_shared_dataloader import collate_fn

loader = DataLoader(dataset_aug, batch_size=4, shuffle=True, num_workers=2, 
                    collate_fn=collate_fn, prefetch_factor=2)
loader_iter = iter(loader)
for i in range(3):
    t0 = time.time()
    batch = next(loader_iter)
    print(f"Batch {i+1}: shape={batch['image'].shape}, time={time.time()-t0:.2f}s")